# 04 — Mining Social Network Graphs in Spark: Girvan-Newman, SimRank, Clustering Coefficient
**Exam mapping (P4):** 4a Girvan-Newman partition + justify K by modularity + largest community [10]; 4b SimRank partition + justify K [10]; 4c global clustering coefficient [5].

**THE key move (worth the most points):** Girvan-Newman is **O(V·E) per edge removal**; SimRank is **O(n²) memory / O(n²d²) per iter**. At ~596K nodes both are infeasible. So you **induce the top-N-by-degree subgraph in Spark, state the complexity reason in a comment, run the exact algorithm on that core**, and say what the subset represents (the high-connectivity core carrying the meaningful community structure). Clustering coefficient (4c) is cheap and runs on the **full** graph in Spark via self-joins.

In [ ]:
# ---- Spark session + load links ----
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import collections, copy
from collections import Counter, deque

spark = (SparkSession.builder.appName("sds-graph-mining")
         .master("local[*]").config("spark.ui.enabled", "false")
         .config("spark.sql.shuffle.partitions", "16").getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

DATA_PATH = "data/pageviews_practice.csv"     # <-- exam: "pageviews.csv"
raw = (spark.read.csv(DATA_PATH, sep="\t", header=False, inferSchema=True)
       .toDF("src", "dst", "type", "count"))
links = raw.filter(F.col("type") == "link").select("src", "dst").dropDuplicates().cache()
print("link edges:", links.count())

## The subsetting cell (run before 4a and 4b) — induce the top-N high-degree core in Spark
Justify N in your write-up: GN/SimRank are super-linear, so we restrict to the top-N nodes by total (in+out) degree — the densely connected core where community structure is concentrated — and run the exact algorithm there.

In [ ]:
N_TOP = 150     # top-N by total degree; keeps GN/SimRank tractable, holds the core structure

out_d = links.groupBy("src").count().withColumnRenamed("src", "node").withColumnRenamed("count", "o")
in_d  = links.groupBy("dst").count().withColumnRenamed("dst", "node").withColumnRenamed("count", "i")
top_nodes = {r["node"] for r in
             (out_d.join(in_d, "node", "outer").fillna(0)
                   .withColumn("tot", F.col("o") + F.col("i"))
                   .orderBy(F.desc("tot")).limit(N_TOP).collect())}

# induce directed edges among the top-N nodes (small enough to bring to driver)
induced = [(r["src"], r["dst"]) for r in
           links.filter(F.col("src").isin(top_nodes) & F.col("dst").isin(top_nodes)).collect()]
print(f"top-{N_TOP} nodes -> {len(induced)} induced directed edges")

# undirected adjacency (GN) and directed in-neighbours (SimRank)
adj = collections.defaultdict(set)
for u, v in induced:
    if u != v: adj[u].add(v); adj[v].add(u)
und_nodes = sorted(adj)
und_edges = [(u, v) for u in adj for v in adj[u] if u < v]
in_nbr = collections.defaultdict(set)
for u, v in induced:
    if u != v: in_nbr[v].add(u)
sr_nodes = sorted({x for e in induced for x in e if e[0] != e[1]})
print(f"undirected: {len(und_nodes)} nodes, {len(und_edges)} edges | simrank nodes: {len(sr_nodes)}")

## 4a — Girvan-Newman (Brandes edge-betweenness + modularity to pick K)
Remove the highest edge-betweenness edge repeatedly; after each removal recompute connected components and **modularity Q = Σ_c [ L_c/m − (d_c/2m)² ]**; the K at **peak Q** is the chosen partition.

In [ ]:
def brandes_eb(adj, nodes):
    "O(VE) edge betweenness (undirected, Brandes)."
    eb = collections.defaultdict(float)
    for s in nodes:
        S, pred = [], {v: [] for v in nodes}
        sigma = dict.fromkeys(nodes, 0.0); sigma[s] = 1.0
        dist = dict.fromkeys(nodes, -1);   dist[s] = 0
        Q = deque([s])
        while Q:
            v = Q.popleft(); S.append(v)
            for w in adj[v]:
                if dist[w] < 0: dist[w] = dist[v] + 1; Q.append(w)
                if dist[w] == dist[v] + 1: sigma[w] += sigma[v]; pred[w].append(v)
        delta = dict.fromkeys(nodes, 0.0)
        while S:
            w = S.pop()
            for v in pred[w]:
                c = (sigma[v] / sigma[w]) * (1.0 + delta[w])
                eb[(min(v, w), max(v, w))] += c; delta[v] += c
    return {e: sc / 2.0 for e, sc in eb.items()}

def components(adj, nodes):
    seen = {}; cid = 0
    for s in nodes:
        if s not in seen:
            Q = deque([s])
            while Q:
                u = Q.popleft()
                if u not in seen:
                    seen[u] = cid
                    for nb in adj.get(u, ()):
                        if nb not in seen: Q.append(nb)
            cid += 1
    return seen

def modularity(orig_edges, comp):
    m = len(orig_edges)
    if m == 0: return 0.0
    deg = collections.defaultdict(int)
    for u, v in orig_edges: deg[u]+=1; deg[v]+=1
    Lc, dc = collections.defaultdict(int), collections.defaultdict(int)
    for u, v in orig_edges:
        if comp.get(u) == comp.get(v): Lc[comp[u]] += 1
    for nd, c in comp.items(): dc[c] += deg.get(nd, 0)
    return sum(Lc.get(c,0)/m - (dc.get(c,0)/(2*m))**2 for c in set(comp.values()))

work = copy.deepcopy(adj)
history = []
for it in range(min(120, len(und_edges))):
    eb = brandes_eb(work, und_nodes)
    if not eb: break
    u, v = max(eb, key=eb.get)
    work[u].discard(v); work[v].discard(u)
    comp = components(work, und_nodes)
    history.append((it+1, max(comp.values())+1, modularity(und_edges, comp), copy.deepcopy(comp)))

best = max(history, key=lambda x: x[2])
best_it, best_K, best_Q, best_comp = best
print(f"Peak modularity Q={best_Q:.5f} at K={best_K} communities (iteration {best_it})")
sizes = Counter(best_comp.values())
lc_id, lc_sz = sizes.most_common(1)[0]
print(f"\nAll {best_K} communities (size): {sorted(sizes.values(), reverse=True)}")
print(f"\nLargest community ({lc_sz} members):")
for nm in sorted(n for n, c in best_comp.items() if c == lc_id): print(" ", nm)

### What to report (4a)
- Complexity justification for the top-N subset (GN is O(VE)/removal). Peak-Q rule for K, with the Q value and K quoted **from the printout** (not a remembered number).
- Largest community membership + one sentence of interpretation (what theme binds it).

## 4b — SimRank (structural similarity → threshold → communities)
`s(a,b) = c/(|I(a)|·|I(b)|) · Σ_{x∈I(a),y∈I(b)} s(x,y)`, `s(a,a)=1`, decay **c=0.8**. Converge, then threshold the similarity matrix at θ and take connected components. Justify θ at a **stable plateau** of the K–θ sweep.

In [ ]:
n = len(sr_nodes); idx = {nd: i for i, nd in enumerate(sr_nodes)}
C = 0.8
S = [[1.0 if i == j else 0.0 for j in range(n)] for i in range(n)]
for it in range(12):
    Snew = [[1.0 if i == j else 0.0 for j in range(n)] for i in range(n)]
    for i, u in enumerate(sr_nodes):
        Iu = list(in_nbr[u])
        if not Iu: continue
        for j, v in enumerate(sr_nodes):
            if i == j: continue
            Iv = list(in_nbr[v])
            if not Iv: continue
            tot = 0.0
            for a in Iu:
                ia = idx[a]
                for b in Iv: tot += S[ia][idx[b]]
            Snew[i][j] = C / (len(Iu)*len(Iv)) * tot
    diff = max(abs(Snew[i][j]-S[i][j]) for i in range(n) for j in range(n))
    S = Snew
    if diff < 1e-5: print(f"converged at iter {it+1}"); break

def sim_comm(theta):
    g = collections.defaultdict(set)
    for i in range(n):
        for j in range(i+1, n):
            if S[i][j] >= theta: g[sr_nodes[i]].add(sr_nodes[j]); g[sr_nodes[j]].add(sr_nodes[i])
    return components(g, sr_nodes)

print(f"{'theta':>6} {'K':>4}")
prev, chosen = None, None
for th in [0.02,0.05,0.10,0.15,0.20,0.30,0.40,0.50]:
    K = max(sim_comm(th).values())+1
    if prev == K and chosen is None: chosen = th
    print(f"{th:>6.2f} {K:>4}{'  <- stable plateau' if prev==K and chosen==th else ''}")
    prev = K
chosen = chosen or 0.05
comp = sim_comm(chosen); sizes = Counter(comp.values())
lc_id, lc_sz = sizes.most_common(1)[0]
print(f"\nChosen theta={chosen} -> K={max(comp.values())+1}; largest community {lc_sz} members:")
for nm in sorted(x for x,c in comp.items() if c == lc_id)[:40]: print(" ", nm)

### What to report (4b)
- c=0.8, convergence iteration, θ chosen at the K–θ plateau. **Honest comparison to 4a:** SimRank thresholding tends to give a more fragmented / less semantically cohesive partition than GN on this graph — say so if that's what you see (it scored on the real exam).

## 4c — Global clustering coefficient (transitivity) on the FULL graph in Spark
`CC = (# closed ordered 2-paths) / (# ordered 2-paths)`. Symmetrize edges, self-join to form 2-hop paths u–v–w (u≠w), then join again to check the closing edge u→w exists. All in Spark, no `collect()`.

In [ ]:
E = (links.select("src", "dst")
        .union(links.select(F.col("dst").alias("src"), F.col("src").alias("dst")))
        .distinct().cache())

two_hop = (E.alias("e1").join(E.alias("e2"), F.col("e1.dst") == F.col("e2.src"))
             .filter(F.col("e1.src") != F.col("e2.dst"))
             .select(F.col("e1.src").alias("u"), F.col("e2.dst").alias("w")))
n_paths = two_hop.count()

n_closed = (two_hop.join(E.alias("e3"),
                (F.col("u") == F.col("e3.src")) & (F.col("w") == F.col("e3.dst")))
                .count())

CC = n_closed / n_paths if n_paths else 0.0
print(f"ordered 2-paths     : {n_paths:,}")
print(f"ordered closed paths: {n_closed:,}")
print(f"undirected triangles: {n_closed // 6:,}")
print(f"GLOBAL CLUSTERING COEFFICIENT = {CC:.6f}")
E.unpersist()

### What to report (4c) + gotchas
- CC value + interpretation: a clickstream is a navigational graph (edges = reader navigation, not social ties), so triadic closure is weak and a **near-zero CC is expected** for a sparse hub-and-spoke graph — say this.
- **Exam gotchas:** GN/SimRank need the complexity-justified subset; K from **peak modularity** (GN) and **θ plateau** (SimRank); clustering coefficient is cheap → run on the **full** graph in Spark; quote every number from the printout, and keep prose numbers consistent with executed output.

In [ ]:
links.unpersist(); spark.stop(); print("done")